[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C49_Encoder_Seq2Seq_Course/03_finetuning/03_finetuning.ipynb)

# 03 · 下游微调三范式（分类头 / BIO 约束解码 / span 联合搜索，全部从零）

目标：把 **序列分类 → token 分类 + Viterbi 约束解码 → span 抽取 + 联合搜索** 从零实现，
每个解码器都**对拍暴力枚举**、每个指标都**手写并暴露它的陷阱**。

路线：三种头 → 池化方式对比 → BIO 合法性与 Viterbi（对拍暴力）→ token级 vs 实体级 F1 →
span 联合搜索三种实现对拍 → EM 归一化的影响 → MCC vs accuracy → ✏️ 练习 → 📖 答案 → 🧪 种子方差胶囊。

> 心智模型：**头很简单，难的是输出空间的结构约束怎么在解码时被强制执行，以及指标是否真的度量了你关心的东西。**

## 1 · 三种头：参数量都可以忽略

预训练主干给你 `h: (n, d)`。三种范式只是用不同的方式把 `h` 变成任务输出。

In [ ]:
import numpy as np, math, re, string, itertools
rng = np.random.default_rng(0)

D, N = 32, 12          # 隐维度、序列长度

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x); return e / e.sum(axis=axis, keepdims=True)

class SequenceClassificationHead:
    def __init__(self, d, n_classes, seed=0):
        r = np.random.default_rng(seed); self.W = r.normal(size=(d, n_classes)) * 0.1
    def __call__(self, h, pooling='cls', attn_mask=None):
        if pooling == 'cls':
            pooled = h[0]
        elif pooling == 'mean':
            m = np.ones(len(h)) if attn_mask is None else attn_mask
            pooled = (h * m[:, None]).sum(0) / m.sum()
        elif pooling == 'max':
            pooled = h.max(0)
        else:
            raise ValueError(pooling)
        return pooled @ self.W

class TokenClassificationHead:
    def __init__(self, d, n_labels, seed=0):
        r = np.random.default_rng(seed); self.W = r.normal(size=(d, n_labels)) * 0.1
    def __call__(self, h):
        return h @ self.W                        # (n, K) 每个位置一组 logits

class SpanHead:
    def __init__(self, d, seed=0):
        r = np.random.default_rng(seed); self.W = r.normal(size=(d, 2)) * 0.1
    def __call__(self, h):
        logits = h @ self.W
        return logits[:, 0], logits[:, 1]        # start_logits, end_logits

h = rng.normal(size=(N, D))
cls_head, tok_head, span_head = SequenceClassificationHead(D, 3), TokenClassificationHead(D, 9), SpanHead(D)
print('序列分类输出:', cls_head(h).shape, '(3 类)')
print('token 分类输出:', tok_head(h).shape, '(每位置 9 标签)')
s_log, e_log = span_head(h)
print('span 输出:', s_log.shape, e_log.shape)

BACKBONE = 110e6
for name, head in [('序列分类(3类)', cls_head.W), ('token分类(9标签)', tok_head.W), ('span抽取', span_head.W)]:
    print(f'{name:<18s} 头参数 {head.size:>6d}  (主干 110M 的 {head.size/BACKBONE*768/D:.5%})')
assert cls_head(h).shape == (3,) and tok_head(h).shape == (N, 9)
print('\n✅ 三种头的参数量都可忽略 —— 微调成本几乎完全由主干决定，与任务类型无关')

### 池化方式：[CLS] 还是 mean？

**要微调就用 [CLS]（梯度会教它汇总），不微调直接抽向量就用 mean pooling。**
下面用一个「未微调」的模拟场景展示差别。

In [ ]:
def make_sentences(n=200, seed=3):
    '''每句由若干「词向量」构成；标签 = 是否含有「关键词」（用一个特定方向表示）。'''
    r = np.random.default_rng(seed)
    key = r.normal(size=D); key /= np.linalg.norm(key)
    Hs, ys = [], []
    for _ in range(n):
        L = r.integers(6, 12)
        hh = r.normal(size=(L, D)) * 0.5
        y = int(r.random() < 0.5)
        if y:
            hh[r.integers(1, L)] += key * 3.0          # 在某个**中间**位置埋入关键信号
        hh = np.vstack([r.normal(size=D) * 0.5, hh])   # 位置 0 是 [CLS]，**未被训练过**
        Hs.append(hh); ys.append(y)
    return Hs, np.array(ys)

def probe(Hs, ys, pooling, seed=0):
    '''用池化向量训一个线性探针，返回准确率。'''
    feats = []
    for hh in Hs:
        if pooling == 'cls':  feats.append(hh[0])
        elif pooling == 'mean': feats.append(hh.mean(0))
        else: feats.append(hh.max(0))
    Xf = np.stack(feats); Xf = (Xf - Xf.mean(0)) / (Xf.std(0) + 1e-8)
    r = np.random.default_rng(seed); w = r.normal(size=D) * 0.01; b = 0.0
    for _ in range(1500):
        p = 1 / (1 + np.exp(-(Xf @ w + b)))
        g = p - ys
        w -= 0.5 * (Xf.T @ g) / len(ys); b -= 0.5 * g.mean()
    return ((Xf @ w + b > 0).astype(int) == ys).mean()

Hs, ys = make_sentences()
for pool in ['cls', 'mean', 'max']:
    print(f'{pool:>5s} pooling 探针准确率: {probe(Hs, ys, pool):.1%}')
acc_cls, acc_mean = probe(Hs, ys, 'cls'), probe(Hs, ys, 'mean')
assert acc_mean > acc_cls + 0.10, '未微调时，mean pooling 应显著优于 [CLS]'
print(f'\n✅ 未微调场景下 mean({acc_mean:.0%}) 远好于 [CLS]({acc_cls:.0%})：')
print('   [CLS] 之所以能代表整句，是因为**有目标在训练它**；没训练过的 [CLS] 什么也不是。')
print('   直接拿预训练 BERT 的 [CLS] 当句向量做检索，是个经典的、效果很差的做法。')

## 2 · BIO 标注：合法性约束与 Viterbi

BIO 标签序列有**语法**：`O → I-X` 非法，`B-PER → I-LOC` 非法。
朴素逐位置 argmax 不知道这些，会产出非法序列。

In [ ]:
ENTITY_TYPES = ['PER', 'LOC', 'ORG']
LABELS = ['O'] + [f'{p}-{t}' for t in ENTITY_TYPES for p in ('B', 'I')]
L2I = {l: i for i, l in enumerate(LABELS)}
K = len(LABELS)
print('标签集:', LABELS)

def legal_transition(prev, cur):
    '''prev -> cur 是否合法（IOB2 规则）。'''
    if cur == 'O' or cur.startswith('B-'):
        return True                                   # O 与 B-X 永远可以开始
    # cur 是 I-X：只能跟在 B-X 或 I-X 之后（同类型）
    t = cur[2:]
    return prev in (f'B-{t}', f'I-{t}')

def transition_matrix():
    T = np.zeros((K, K))
    for i, p in enumerate(LABELS):
        for j, c in enumerate(LABELS):
            if not legal_transition(p, c):
                T[i, j] = -1e9                        # 屏蔽非法转移
    return T

def start_mask():
    '''序列开头不能是 I-X。'''
    m = np.zeros(K)
    for i, l in enumerate(LABELS):
        if l.startswith('I-'): m[i] = -1e9
    return m

T, S0 = transition_matrix(), start_mask()
assert legal_transition('B-PER', 'I-PER') and not legal_transition('B-PER', 'I-LOC')
assert not legal_transition('O', 'I-LOC')
assert legal_transition('I-LOC', 'B-PER')
n_legal = (T > -1).sum()
print(f'\n{K}×{K} = {K*K} 种转移中，合法的有 {n_legal} 种（{n_legal/K/K:.0%}）')
print('✅ 转移矩阵正确：非法转移被屏蔽为 -1e9')

In [ ]:
def naive_decode(emissions):
    '''逐位置 argmax —— 不管约束。'''
    return [LABELS[i] for i in emissions.argmax(1)]

def viterbi_decode(emissions, T, S0):
    '''O(n·K²) 动态规划，返回最优合法路径。'''
    n = len(emissions)
    delta = emissions[0] + S0
    back = np.zeros((n, K), dtype=int)
    for i in range(1, n):
        scores = delta[:, None] + T                  # (K_prev, K_cur)
        back[i] = scores.argmax(0)
        delta = emissions[i] + scores.max(0)
    path = [int(delta.argmax())]
    for i in range(n - 1, 0, -1):
        path.append(int(back[i][path[-1]]))
    return [LABELS[i] for i in reversed(path)]

def brute_force_decode(emissions, T, S0):
    '''暴力枚举所有 K^n 条路径（只在极小 n 下用，作为对拍参考）。'''
    n = len(emissions)
    best, best_score = None, -np.inf
    for path in itertools.product(range(K), repeat=n):
        sc = emissions[0][path[0]] + S0[path[0]]
        for i in range(1, n):
            sc += T[path[i-1], path[i]] + emissions[i][path[i]]
        if sc > best_score:
            best, best_score = path, sc
    return [LABELS[i] for i in best]

# 对拍：小规模下 Viterbi 必须与暴力枚举给出相同的最优路径
for trial in range(20):
    em = np.random.default_rng(trial).normal(size=(4, K))
    v, b = viterbi_decode(em, T, S0), brute_force_decode(em, T, S0)
    assert v == b, f'trial {trial}: Viterbi {v} != 暴力 {b}'
print(f'✅ 对拍通过：20 组随机发射分数下，Viterbi(O(nK²)) 与暴力枚举(O(K^n)) 结果完全一致')
print(f'   规模对比 n=4: Viterbi {4*K*K} 次操作 vs 暴力 {K**4:,} 条路径')

In [ ]:
def count_illegal(labels):
    bad = 0
    if labels and labels[0].startswith('I-'): bad += 1
    for a, b in zip(labels, labels[1:]):
        if not legal_transition(a, b): bad += 1
    return bad

n_seq, tot_naive, tot_vit = 300, 0, 0
for s in range(n_seq):
    em = np.random.default_rng(s).normal(size=(15, K)) * 1.2
    tot_naive += count_illegal(naive_decode(em))
    tot_vit   += count_illegal(viterbi_decode(em, T, S0))
print(f'{n_seq} 条序列（各 15 token）:')
print(f'  朴素 argmax : {tot_naive} 处非法转移')
print(f'  Viterbi     : {tot_vit} 处非法转移')
assert tot_naive > 0, '朴素解码必然产生非法转移'
assert tot_vit == 0, 'Viterbi 必须保证零非法转移'
print('\n✅ 约束解码是**零成本的正确性保证**：O(nK²)=15×81≈1200 次操作，')
print('   相比一次 BERT 前向的 ~1e10 FLOPs 完全可以忽略。没有理由不做。')

### token 级 F1 会系统性高估：必须用实体级

In [ ]:
def extract_entities(labels):
    '''从 BIO 序列抽出实体集合 {(类型, 起, 止)}。'''
    ents, start, typ = set(), None, None
    for i, l in enumerate(labels + ['O']):
        if l.startswith('B-') or l == 'O' or (l.startswith('I-') and typ != l[2:]):
            if start is not None:
                ents.add((typ, start, i - 1)); start, typ = None, None
        if l.startswith('B-'):
            start, typ = i, l[2:]
        elif l.startswith('I-') and start is None:
            start, typ = i, l[2:]        # 容错：非法的 I-X 开头当作 B-X
    return ents

def token_f1(gold, pred):
    tp = sum(g == p != 'O' for g, p in zip(gold, pred))
    n_g = sum(g != 'O' for g in gold); n_p = sum(p != 'O' for p in pred)
    prec = tp / n_p if n_p else 0.0; rec = tp / n_g if n_g else 0.0
    return 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)

def entity_f1(gold, pred):
    g, p = extract_entities(gold), extract_entities(pred)
    tp = len(g & p)
    prec = tp / len(p) if p else 0.0; rec = tp / len(g) if g else 0.0
    return 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)

gold = ['B-PER', 'I-PER', 'O', 'B-LOC', 'I-LOC', 'O']
pred = ['B-PER', 'I-PER', 'O', 'B-LOC', 'O',     'O']    # 「北京」的第二个 token 标错
print('金标:', gold)
print('预测:', pred)
print(f'token 级 F1 : {token_f1(gold, pred):.1%}')
print(f'实体级 F1   : {entity_f1(gold, pred):.1%}')
print(f'金标实体: {sorted(extract_entities(gold))}')
print(f'预测实体: {sorted(extract_entities(pred))}')
assert token_f1(gold, pred) > entity_f1(gold, pred), 'token 级会高估'
assert entity_f1(gold, pred) < 0.7, '一个实体边界错 -> 该实体完全不算对'
print('\n✅ 「一半正确」在实体级评估里等于**完全错误** —— 用户要的是「北京」，不是半个。')

## 3 · span 抽取：联合搜索的三种实现

分别取两个 argmax 会产生 `end < start`、跨越 [SEP]、答案过长三类无效答案。
**必须联合搜索。** 三种实现，两两对拍。

In [ ]:
def naive_argmax_span(s_log, e_log):
    return int(s_log.argmax()), int(e_log.argmax())

def joint_search_quadratic(s_log, e_log, ctx_start, ctx_end, max_len=10):
    '''O(n²) 暴力：枚举所有合法 (i,j)。'''
    best, bs = (ctx_start, ctx_start), -np.inf
    for i in range(ctx_start, ctx_end + 1):
        for j in range(i, min(i + max_len, ctx_end) + 1):
            sc = s_log[i] + e_log[j]
            if sc > bs: bs, best = sc, (i, j)
    return best

def joint_search_window(s_log, e_log, ctx_start, ctx_end, max_len=10):
    '''O(n·max_len)：对每个 i 只看长度窗口内的 j。'''
    best, bs = (ctx_start, ctx_start), -np.inf
    for i in range(ctx_start, ctx_end + 1):
        hi = min(i + max_len, ctx_end)
        j = i + int(np.argmax(e_log[i:hi + 1]))
        sc = s_log[i] + e_log[j]
        if sc > bs: bs, best = sc, (i, j)
    return best

n_tok, CTX_S, CTX_E = 24, 6, 22       # 0..5 是问题+[SEP]，6..22 是上下文
bad_cases = 0
for trial in range(200):
    r = np.random.default_rng(trial)
    s_log, e_log = r.normal(size=n_tok), r.normal(size=n_tok)
    i0, j0 = naive_argmax_span(s_log, e_log)
    if j0 < i0 or not (CTX_S <= i0 <= CTX_E) or not (CTX_S <= j0 <= CTX_E) or j0 - i0 > 10:
        bad_cases += 1
    a = joint_search_quadratic(s_log, e_log, CTX_S, CTX_E)
    b = joint_search_window(s_log, e_log, CTX_S, CTX_E)
    assert a == b, f'trial {trial}: 两种联合搜索应给出相同结果 {a} vs {b}'
    assert CTX_S <= a[0] <= a[1] <= CTX_E and a[1] - a[0] <= 10, '联合搜索必须满足全部约束'
print(f'✅ 两种联合搜索实现对拍通过（200 组随机 logits）')
print(f'⚠️  朴素双 argmax 在 {bad_cases}/200 = {bad_cases/200:.0%} 的情况下产生**无效答案**')
assert bad_cases > 100, '朴素做法应频繁违反约束'
print('   （end<start、落进问题区、或答案过长）')

### 不可回答（SQuAD 2.0）：用 [CLS] 作为 (0,0) span

In [ ]:
def predict_with_null(s_log, e_log, ctx_start, ctx_end, max_len=10, null_threshold=0.0):
    '''返回 (span 或 None, 差值分数)。'''
    i, j = joint_search_window(s_log, e_log, ctx_start, ctx_end, max_len)
    best_span_score = s_log[i] + e_log[j]
    null_score = s_log[0] + e_log[0]              # [CLS] 位置
    diff = best_span_score - null_score
    return ((i, j) if diff > null_threshold else None), diff

r = np.random.default_rng(7)
s_log, e_log = r.normal(size=n_tok), r.normal(size=n_tok)
s_log[10] += 4; e_log[12] += 4                    # 制造一个明显的答案
span, diff = predict_with_null(s_log, e_log, CTX_S, CTX_E)
print(f'有明显答案时: span={span}, diff={diff:.2f}')
assert span == (10, 12), f'应抽出 (10,12)，得到 {span}'

s2, e2 = r.normal(size=n_tok) * 0.3, r.normal(size=n_tok) * 0.3
s2[0] += 5; e2[0] += 5                            # [CLS] 分数最高 = 无法回答
span2, diff2 = predict_with_null(s2, e2, CTX_S, CTX_E)
print(f'无答案时:     span={span2}, diff={diff2:.2f}')
assert span2 is None, '应判定为不可回答'
print('\n✅ 不需要额外的分类头 —— 复用同一套 logits，把 [CLS] 当作「无答案」span。')
print('   阈值 null_threshold 需要在验证集上调（两类分数尺度不同）。')

### EM 归一化：忘了它会低估 5-10 个点

In [ ]:
def normalize_answer(s):
    '''SQuAD 官方归一化：小写 -> 去标点 -> 去冠词 -> 压空格。'''
    s = s.lower()
    s = ''.join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return ' '.join(s.split())

def exact_match(pred, gold, normalize=True):
    f = normalize_answer if normalize else (lambda x: x)
    return int(f(pred) == f(gold))

def token_overlap_f1(pred, gold):
    p, g = normalize_answer(pred).split(), normalize_answer(gold).split()
    common = {}
    for t in p:
        if t in g: common[t] = min(p.count(t), g.count(t))
    tp = sum(common.values())
    if tp == 0: return 0.0
    prec, rec = tp / len(p), tp / len(g)
    return 2 * prec * rec / (prec + rec)

pairs = [('the White House', 'White House'),
         ('Barack Obama.', 'Barack Obama'),
         ('An apple', 'apple'),
         ('New  York', 'New York'),
         ('Paris', 'London')]
print(f"{'预测':<20s} {'金标':<16s} {'EM(无归一)':>11s} {'EM(归一)':>9s} {'F1':>6s}")
em_raw = em_norm = 0
for p, g in pairs:
    a, b = exact_match(p, g, False), exact_match(p, g, True)
    em_raw += a; em_norm += b
    print(f'{p:<20s} {g:<16s} {a:>11d} {b:>9d} {token_overlap_f1(p,g):>6.2f}')
print(f'\nEM 无归一化 {em_raw}/{len(pairs)} = {em_raw/len(pairs):.0%}')
print(f'EM 有归一化 {em_norm}/{len(pairs)} = {em_norm/len(pairs):.0%}')
assert em_norm > em_raw, '归一化会显著提高 EM'
assert exact_match('Paris', 'London') == 0, '真错的仍然是 0'
print(f'\n✅ 忘记归一化会让你低估 {(em_norm-em_raw)/len(pairs):.0%} —— 然后去做一堆无用的优化。')

## 4 · 指标陷阱：accuracy 0.95 而 MCC 0.00

In [ ]:
def confusion(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum()); tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum()); fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return tp, tn, fp, fn

def mcc(y_true, y_pred):
    tp, tn, fp, fn = confusion(y_true, y_pred)
    num = tp * tn - fp * fn
    den = math.sqrt((tp+fp) * (tp+fn) * (tn+fp) * (tn+fn))
    return 0.0 if den == 0 else num / den

def macro_f1(y_true, y_pred):
    fs = []
    for c in (0, 1):
        tp = int(((y_true == c) & (y_pred == c)).sum())
        fp = int(((y_true != c) & (y_pred == c)).sum())
        fn = int(((y_true == c) & (y_pred != c)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        fs.append(0.0 if p + r == 0 else 2 * p * r / (p + r))
    return float(np.mean(fs))

r = np.random.default_rng(0)
y = (r.random(2000) < 0.05).astype(int)       # 5% 正例（毒性检测的典型分布）
pred_majority = np.zeros_like(y)              # 「全预测多数类」的退化模型
pred_decent   = y.copy()
flip = r.choice(len(y), size=60, replace=False); pred_decent[flip] ^= 1   # 一个真的还行的模型

print(f"{'模型':<16s} {'accuracy':>9s} {'macro-F1':>9s} {'MCC':>7s}")
for name, p in [('全预测多数类', pred_majority), ('真的还行的模型', pred_decent)]:
    acc = (p == y).mean()
    print(f'{name:<16s} {acc:>9.1%} {macro_f1(y,p):>9.3f} {mcc(y,p):>7.3f}')

assert (pred_majority == y).mean() > 0.94, '全预测多数类的 accuracy 高达 95%'
assert mcc(y, pred_majority) == 0.0, 'MCC 正确识别出「什么也没学到」'
assert macro_f1(y, pred_majority) < 0.5, 'macro-F1 也能识别'
assert mcc(y, pred_decent) > 0.5, '真正有用的模型 MCC 显著为正'
print('\n✅ accuracy 95% 而 MCC 0.00 —— 模型什么也没学到，但指标看起来很棒。')
print('   不平衡任务必须看 MCC 或 macro-F1（GLUE 的 CoLA 用 MCC 正是这个理由）。')

## ✏️ 练习 1：判别式学习率

实现 `layerwise_lr(base_lr, n_layers, gamma=0.95)`：返回长度 `n_layers+1` 的列表
（索引 0 = 嵌入层，索引 n_layers = 顶层），顶层为 `base_lr`，每往下一层乘 `gamma`。

In [ ]:
def layerwise_lr(base_lr, n_layers, gamma=0.95):
    # TODO: lr[l] = base_lr * gamma^(n_layers - l)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
lrs = layerwise_lr(3e-5, 12, gamma=0.95)
assert len(lrs) == 13
assert abs(lrs[-1] - 3e-5) < 1e-12, '顶层应等于 base_lr'
assert lrs == sorted(lrs), '越往下学习率越小'
assert abs(lrs[0] - 3e-5 * 0.95 ** 12) < 1e-12
print(f'顶层 lr {lrs[-1]:.2e} | 嵌入层 lr {lrs[0]:.2e} | 比值 {lrs[0]/lrs[-1]:.2f}')
# gamma=1.0 退化为均一学习率
assert all(abs(x - 3e-5) < 1e-12 for x in layerwise_lr(3e-5, 12, gamma=1.0))
# 极端：gamma=0 等价于冻结除顶层外的全部层
frozen = layerwise_lr(3e-5, 12, gamma=0.0)
assert frozen[-1] == 3e-5 and all(x == 0 for x in frozen[:-1])
print('✅ 练习 1 通过：gamma=1 是均一微调，gamma=0 是只训顶层，中间是一整条谱系')

## ✏️ 练习 2：子词标签对齐

tokenizer 把「Washington」切成 `['Wash','##ing','##ton']`，但标注只有一个 `B-LOC`。
实现 `align_labels(word_ids, word_labels)`：
`word_ids` 是每个 token 对应的**词索引**（`None` 表示特殊 token）。
规则：每个词的**第一个**子词取该词的标签，其余子词与特殊 token 都设为 `-100`。

In [ ]:
def align_labels(word_ids, word_labels):
    # TODO: 遍历 word_ids，记录上一个 word_id；首次出现取标签，重复出现或 None -> -100
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
#  [CLS]  Wash  ##ing  ##ton   is    nice  [SEP]
wids = [None,   0,     0,     0,     1,    2,    None]
wlabs = [L2I['B-LOC'], L2I['O'], L2I['O']]
out = align_labels(wids, wlabs)
assert out == [-100, L2I['B-LOC'], -100, -100, L2I['O'], L2I['O'], -100], out
assert out.count(-100) == 4, '2 个特殊 token + 2 个后续子词'
# 边界：全是特殊 token
assert align_labels([None, None], []) == [-100, -100]
# 每个词恰好一个非 -100
non_ignored = [i for i, v in enumerate(out) if v != -100]
assert len(non_ignored) == len(wlabs), '每个词只贡献一个训练信号'
print('对齐结果:', out)
print('✅ 练习 2 通过：忘了这一步，模型会学到把实体的后续子词标成 O，实体级 F1 莫名很低')

## ✏️ 练习 3：O(n) 的 span 联合搜索

用**后缀最大值**把联合搜索从 O(n·max_len) 降到 O(n)：
预计算 `suf_val[i] = max_{j>=i} e_log[j]` 与对应的 `suf_idx[i]`，
然后对每个 `i` 直接查 `suf_idx[i]`（不带长度上限的版本）。

实现 `joint_search_linear(s_log, e_log, ctx_start, ctx_end)`，返回 `(i, j)`。

In [ ]:
def joint_search_linear(s_log, e_log, ctx_start, ctx_end):
    # TODO: 从右往左算后缀最大值及其索引；再从左往右对每个 i 取 s_log[i] + suf_val[i]
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测：与不带长度上限的 O(n²) 版本对拍 ——
def joint_search_quadratic_nolimit(s_log, e_log, cs, ce):
    best, bs = (cs, cs), -np.inf
    for i in range(cs, ce + 1):
        for j in range(i, ce + 1):
            if s_log[i] + e_log[j] > bs:
                bs, best = s_log[i] + e_log[j], (i, j)
    return best

for trial in range(200):
    r = np.random.default_rng(1000 + trial)
    s_, e_ = r.normal(size=n_tok), r.normal(size=n_tok)
    a = joint_search_linear(s_, e_, CTX_S, CTX_E)
    b = joint_search_quadratic_nolimit(s_, e_, CTX_S, CTX_E)
    assert a == b, f'trial {trial}: 线性 {a} != 二次 {b}'
    assert CTX_S <= a[0] <= a[1] <= CTX_E
print('✅ 练习 3 通过：200 组对拍一致，复杂度从 O(n²) 降到 O(n)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def layerwise_lr(base_lr, n_layers, gamma=0.95):
    return [base_lr * (gamma ** (n_layers - l)) for l in range(n_layers + 1)]

In [ ]:
# 练习 2 参考答案
def align_labels(word_ids, word_labels):
    out, prev = [], None
    for wid in word_ids:
        if wid is None:        out.append(-100)
        elif wid != prev:      out.append(word_labels[wid])
        else:                  out.append(-100)
        prev = wid
    return out

In [ ]:
# 练习 3 参考答案
def joint_search_linear(s_log, e_log, ctx_start, ctx_end):
    n = ctx_end - ctx_start + 1
    suf_val = np.empty(n); suf_idx = np.empty(n, dtype=int)
    suf_val[-1], suf_idx[-1] = e_log[ctx_end], ctx_end
    for k in range(n - 2, -1, -1):
        pos = ctx_start + k
        if e_log[pos] >= suf_val[k + 1]:
            suf_val[k], suf_idx[k] = e_log[pos], pos
        else:
            suf_val[k], suf_idx[k] = suf_val[k + 1], suf_idx[k + 1]
    best, bs = (ctx_start, ctx_start), -np.inf
    for k in range(n):
        i = ctx_start + k
        sc = s_log[i] + suf_val[k]
        if sc > bs:
            bs, best = sc, (i, int(suf_idx[k]))
    return best

---
## 🧪 真实数据胶囊：种子方差 —— 你的「涨了 1 分」可能是噪声

Dodge et al. 2020 发现：GLUE 小数据集上，**仅改随机种子**，BERT 微调分数波动可达 2–3 分，
甚至有些种子会训崩。下面模拟这个现象，并计算「要多少种子才能可靠地检测出 1 分的提升」。

In [ ]:
def simulate_finetune(true_score, seed, n_train=2500, collapse_prob=0.06):
    '''模拟一次微调：真实能力 true_score，加上种子噪声，小概率训崩。'''
    r = np.random.default_rng(seed)
    if r.random() < collapse_prob:
        return 52.0 + r.normal(0, 0.5)            # 训崩：退化到多数类基线
    noise_sd = 30.0 / math.sqrt(n_train)          # 小数据集噪声更大
    return true_score + r.normal(0, noise_sd)

BASE_TRUE, NEW_TRUE = 78.0, 79.0                  # 新方法真实高 1 分
runs_base = [simulate_finetune(BASE_TRUE, s) for s in range(20)]
runs_new  = [simulate_finetune(NEW_TRUE, 1000 + s) for s in range(20)]
print(f'baseline 20 个种子: 均值 {np.mean(runs_base):.2f} ± {np.std(runs_base):.2f}, '
      f'范围 [{min(runs_base):.1f}, {max(runs_base):.1f}]')
print(f'新方法  20 个种子: 均值 {np.mean(runs_new):.2f} ± {np.std(runs_new):.2f}, '
      f'范围 [{min(runs_new):.1f}, {max(runs_new):.1f}]')

# 单次实验有多大概率给出**错误**的结论？
wrong = sum(1 for a, b in zip(runs_base, runs_new) if a > b)
print(f'\n单种子对比: {wrong}/20 = {wrong/20:.0%} 的情况下 baseline 反而「赢了」')
assert wrong > 0, '单次实验必然有相当概率给出错误结论'
assert np.std(runs_base) > 0.5, '种子方差应显著'
print('⚠️  也就是说，只跑一个种子，你有相当概率得出完全相反的结论。')

**🧪 胶囊练习**：实现 `seeds_needed(effect_size, noise_sd, power=0.8, alpha=0.05)`：
用双样本 t 检验的经典近似 `n ≈ 2(z_{α/2} + z_β)² σ² / Δ²`，
返回**每组需要的种子数**（向上取整，至少 2）。取 `z_{0.025}=1.96`, `z_{0.2}=0.84`。

In [ ]:
def seeds_needed(effect_size, noise_sd, power=0.8, alpha=0.05):
    # TODO: n = ceil(2 * (1.96 + 0.84)^2 * noise_sd^2 / effect_size^2)，下限 2
    raise NotImplementedError

In [ ]:
# 自测
sd = float(np.std(runs_base))
n1 = seeds_needed(1.0, sd)
n3 = seeds_needed(3.0, sd)
assert n1 >= 2 and n3 >= 2
assert n1 > n3, '效应越小，需要的种子越多'
assert abs(n1 - math.ceil(2 * (1.96 + 0.84) ** 2 * sd ** 2 / 1.0)) <= 1
print(f'噪声标准差 {sd:.2f} 分:')
for eff in [0.5, 1.0, 2.0, 3.0]:
    print(f'  要可靠检测 {eff:.1f} 分的提升 -> 每组需要 {seeds_needed(eff, sd):>3d} 个种子')
print('\n✅ 胶囊练习通过：**「涨了 1 分」通常需要十几个种子才能站得住**。')
print('   工程建议：至少 3-5 个种子，报告均值±标准差，用配对检验判显著性（C03/C10）。')

In [ ]:
# 📖 胶囊参考答案
def seeds_needed(effect_size, noise_sd, power=0.8, alpha=0.05):
    z_a, z_b = 1.96, 0.84
    n = 2 * (z_a + z_b) ** 2 * noise_sd ** 2 / (effect_size ** 2)
    return max(2, math.ceil(n))

---
## 🔧 旁注：真实库里这些对应什么

- **三种头** → `BertForSequenceClassification` / `BertForTokenClassification` / `BertForQuestionAnswering`。
- **子词对齐** → `tokenizer(..., is_split_into_words=True)` + `encoding.word_ids()`；这正是练习 2 的输入。
- **约束解码** → `torchcrf` / `pytorch-crf` 的 `CRF.decode()`；或自己写 Viterbi（就是本 notebook 这段）。
- **实体级 F1** → `seqeval.metrics.f1_score`（**不要**用 sklearn 的 token 级 F1）。
- **span 联合搜索 + 归一化 EM** → HF 的 `squad_v2` metric 与 `postprocess_qa_predictions`（含滑窗合并）。
- **判别式学习率** → 给 optimizer 传 `param_groups`，按层名分组设不同 `lr`。
- **多种子** → `Trainer(args=TrainingArguments(seed=...))`，跑 N 次取均值；`transformers` 的 `set_seed()`。

怎么用这些库真正跑起来，见 **C50**。

### 小结
- 三种范式的**头都可忽略不计**；难的是**输出空间的结构约束**：BIO 的合法转移、span 的 `end≥start`。
- **要微调用 [CLS]，不微调抽向量用 mean pooling**——[CLS] 只有被训练过才有意义。
- **Viterbi 约束解码是零成本的正确性保证**（O(nK²) 对比一次前向的 1e10 FLOPs），且已与暴力枚举对拍。
- **实体级 F1 才是产品指标**；token 级会系统性高估。**EM 必须先归一化**，否则低估 5-10 个点。
- **不平衡任务看 MCC / macro-F1**：accuracy 95% 而 MCC 0.00 的模型什么也没学到。
- **子词只在首片计损失**（-100 忽略其余），忘了这步 F1 会莫名很低。
- **种子方差 2-3 分**：单次实验有相当概率给出相反结论；「涨 1 分」需要十几个种子才站得住。

下一站：**模块 04 · Encoder-Decoder** —— 输出不再是标签或片段，而是要**生成**一段新文本。